<a href="https://colab.research.google.com/github/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

Цель: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

Данные:
- **[`music_composer_genre.csv`](https://github.com/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha/blob/main/data/music_composer_genre.csv) — произведения, авторы, жанры, страны и продолжительность музыкальных композиций**

Что мы делаем:
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [3]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

# **ИЗМЕНЕНО: имя репозитория и URL**
repo = "python-ai-Moiseeva-Sasha"  # было: "python-ai-template"
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    # **ИЗМЕНЕНО: URL репозитория**
    !git clone -q https://github.com/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha.git

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Moiseeva-Sasha


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [4]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

# **ИЗМЕНЕНО: путь к файлу и имя переменной**
df_music = pd.read_csv("data/music_composer_genre.csv")  # было: "data/examples/cartoons_genre_country_duration.csv"

# **ИЗМЕНЕНО: убран второй DataFrame, т.к. у вас один файл**
# df_assessment = pd.read_csv("data/examples/cartoons_assessment_reviews.csv")

print("✅ Загружено строк в df_music:", len(df_music))  # было: df_genre
# print("✅ Загружено строк в df_assessment:", len(df_assessment))  # удалено

✅ Загружено строк в df_music: 902


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `music_composer_genre.csv` есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `work` с URL (ссылкой на объект Wikidata) — **удаляем**, так как в базовом анализе он не требуется.
- Столбцы `workLabel`, `authorLabel`, `genreLabel`, `countryLabel`, `durationLabel` содержат читаемые подписи (названия).

В этом шаге мы:
- **удаляем** столбец с URL Wikidata (`work`);
- переименовываем столбцы, убирая постфикс `Label`:
  - `workLabel` → `work`
  - `authorLabel` → `author`
  - `genreLabel` → `genre`
  - `countryLabel` → `country`
  - `durationLabel` → `duration`
- приводим столбец `duration` к числовому типу, заменяя пропуски на 0.

🔁 **Идемпотентность**: код проверяет текущую структуру данных и пропускает шаги, если они уже выполнены. Это значит, что ячейку можно запускать многократно без ошибок и без порчи данных.

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

In [9]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для df_music
# 🔁 Идемпотентно: можно запускать многократно без ошибок

def clean_music_dataframe(df):
    """
    Очищает и переименовывает столбцы в DataFrame.
    Работает корректно при повторных запусках.
    """
    # === Проверка: данные уже очищены? ===
    expected_clean_cols = {"work", "author", "genre", "country", "duration"}
    current_cols = set(df.columns)

    # Если все ожидаемые "чистые" колонки уже есть — пропускаем
    if expected_clean_cols.issubset(current_cols):
        # Дополнительно убедимся, что "грязных" колонок точно нет
        label_cols_present = [c for c in current_cols if c.endswith("Label")]
        if not label_cols_present and "work" in current_cols and df["work"].dtype == "object" and len(df) > 0:
            # Проверяем, что work — это не URL, а уже название произведения
            sample = str(df["work"].iloc[0]).strip()
            if not sample.startswith("http"):
                print("⏭️ df_music уже очищен, пропускаем преобразования")
                return df

    # === Выполняем очистку ===
    df_clean = df.copy()  # работаем с копией, чтобы не ломать оригинал при ошибке

    # 1) Удаляем столбец с URL, если он ещё есть (начинается с http)
    if "work" in df_clean.columns:
        # Проверяем, содержит ли столбец URL-адреса
        sample_val = str(df_clean["work"].iloc[0]).strip() if len(df_clean) > 0 else ""
        if sample_val.startswith("http://") or sample_val.startswith("https://"):
            df_clean = df_clean.drop(columns=["work"])
            print("🗑️ Удалён столбец 'work' с URL Wikidata")

    # 2) Переименовываем столбцы *Label → без постфикса (если они ещё есть)
    rename_map = {
        "workLabel": "work",
        "authorLabel": "author",
        "genreLabel": "genre",
        "countryLabel": "country",
        "durationLabel": "duration",
    }
    # Переименовываем только те колонки, которые реально существуют
    existing_renames = {k: v for k, v in rename_map.items() if k in df_clean.columns}
    if existing_renames:
        df_clean = df_clean.rename(columns=existing_renames)
        print(f"✏️ Переименованы столбцы: {list(existing_renames.keys())}")

    # 3) Приводим duration к числовому типу (только если столбец есть и не числовой)
    if "duration" in df_clean.columns and not pd.api.types.is_numeric_dtype(df_clean["duration"]):
        df_clean["duration"] = pd.to_numeric(
            df_clean["duration"], errors="coerce"
        ).fillna(0).astype(int)
        print("🔢 Столбец 'duration' преобразован в int")
    elif "duration" in df_clean.columns:
        print("✅ Столбец 'duration' уже числовой")

    return df_clean

# === Применяем функцию ===
df_music = clean_music_dataframe(df_music)

print("\n✅ Данные готовы к анализу")
print(f"📋 Итоговые колонки: {df_music.columns.tolist()}")

⏭️ df_music уже очищен, пропускаем преобразования

✅ Данные готовы к анализу
📋 Итоговые колонки: ['work', 'author', 'genre', 'country', 'duration']


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame `df_music`:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым столбцам (`duration`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код.

🔁 **Идемпотентность**: функция безопасно работает при повторных запусках — проверяет наличие столбцов перед выводом статистики.

In [10]:
# 🔍 Шаг 3. Обзор данных: структура и первые строки

def show_info(df, name, n=5):
    """
    Краткий обзор DataFrame: имя, размер, список столбцов, первые строки и статистика.
    🔁 Идемпотентно: безопасно при повторных запусках.
    """
    print(f"\n📊 {name}")
    print("=" * 50)
    print("📏 Размер:", df.shape, f"({df.shape[0]} строк, {df.shape[1]} столбцов)")
    print("📋 Столбцы:", ", ".join(df.columns))

    print("\n🔍 Первые строки:")
    display(df.head(n))

    # 🔢 Базовая статистика только по числовым столбцам
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        print(f"\n📈 Статистика по числовым столбцам ({', '.join(numeric_cols)}):")
        display(df[numeric_cols].describe())
    else:
        print("\n⚠️ Нет числовых столбцов для статистики")

    # ❓ Информация о пропусках
    null_counts = df.isnull().sum()
    if null_counts.any():
        print("\n❓ Пропущенные значения:")
        print(null_counts[null_counts > 0])
    else:
        print("\n✅ Пропущенных значений нет")

# === Применяем функцию к вашему DataFrame ===
show_info(df_music, "Музыкальные произведения: авторы, жанры, страны, продолжительность (df_music)")


📊 Музыкальные произведения: авторы, жанры, страны, продолжительность (df_music)
📏 Размер: (902, 5) (902 строк, 5 столбцов)
📋 Столбцы: work, author, genre, country, duration

🔍 Первые строки:


,work,author,genre,country,duration
0,Den Nith wohl hinab,Роберт Бёрнс,NaN,Королевство Великобритания,0
1,Q19525966,Карел Тума,NaN,Австро-Венгрия,0
2,Cry of freedom,Брюс Смит,Palingsound,Шотландия,0
3,Wet Day in September,Werner Theunissen,кантри-рок,Королевство Нидерландов,0
4,Smile,Werner Theunissen,кантри-рок,Королевство Нидерландов,0



📈 Статистика по числовым столбцам (duration):


,duration
count,902.000000
mean,26.182927
std,71.805963
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,323.000000



❓ Пропущенные значения:
genre      507
country    184
dtype: int64


## 📊 [5A] Топ-10 самых продуктивных авторов

В этом блоке мы детально анализируем вклад каждого автора:

- **Базовый рейтинг**: сколько произведений каждого автора представлено в датасете;
- **Расширенные метрики** для топ-10 авторов:
  - 🎭 Уникальных жанров: в скольких разных жанрах работал автор;
  - 🌍 Уникальных стран: в скольких странах представлены его работы;
  - ⏱️ Средняя и медианная длительность: характерная продолжительность произведений;
  - 🔗 Всего комбинаций `жанр + страна`: насколько разнообразен репертуар.

🔁 **Идемпотентность**: код проверяет наличие столбцов и корректно обрабатывает пропуски.

---

## 🌍 [5B] Страна-жанр матрица: география музыкальных стилей

Здесь мы исследуем связь между страной и жанром:

- **Кросс-таблица**: сколько произведений каждого жанра создано в каждой стране;
- **Доминирующий жанр по странам**: какой жанр чаще всего встречается в каждой стране;
- **Глобальность жанров**: в скольких странах представлен каждый жанр (Топ-10 "международных" жанров);
- **Разнообразие по странам**: в какой стране представлено наибольшее количество разных жанров;
- **Локальные жанры**: какие жанры встречаются только в одной стране (Топ-10).

> 💡 Эта матрица помогает ответить на вопросы:
> - "Какие жанры характерны для Великобритании?"
> - "Есть ли жанры, которые встречаются по всему миру?"
> - "В какой стране самое разнообразное музыкальное наследие?"

🔁 **Идемпотентность**: все расчёты безопасны при повторных запусках и корректно работают с пропусками.

In [12]:
# 📊 Шаг 5A. Топ-10 самых продуктивных авторов

print("🎵 Анализ продуктивности авторов")
print("=" * 70)

# === Базовый рейтинг: количество произведений по авторам ===
author_counts = df_music["author"].value_counts().head(10)

print("\n🏆 Топ-10 самых продуктивных авторов (по числу произведений):")
for rank, (author, count) in enumerate(author_counts.items(), 1):
    print(f"   {rank}. {author}: {count} произведений")

# === Расширенные метрики для топ-авторов ===
print("\n🔍 Детальная статистика для топ-10 авторов:")
print("-" * 70)

# Создаём сводную таблицу с расширенными метриками
def get_author_stats(group):
    return pd.Series({
        "works_count": len(group),
        "unique_genres": group["genre"].nunique(),
        "unique_countries": group["country"].nunique(),
        "avg_duration": group["duration"].mean(),
        "median_duration": group["duration"].median(),
        "genre_country_combos": group.groupby(["genre", "country"]).ngroups
    })

# Группируем по автору и считаем метрики
author_stats = df_music.groupby("author").apply(get_author_stats).reset_index()

# Фильтруем топ-10 по количеству работ и сортируем
top_authors = author_stats.nlargest(10, "works_count")[
    ["author", "works_count", "unique_genres", "unique_countries",
     "avg_duration", "median_duration", "genre_country_combos"]
].round(2)

# Красивый вывод
for _, row in top_authors.iterrows():
    print(f"\n✍️  {row['author']}")
    print(f"   📚 Произведений: {int(row['works_count'])}")
    print(f"   🎭 Жанров: {int(row['unique_genres'])} | 🌍 Стран: {int(row['unique_countries'])}")
    print(f"   ⏱️ Длительность: ср. {row['avg_duration']:.1f}, медиана {row['median_duration']:.1f}")
    print(f"   🔗 Уникальных комбинаций жанр+страна: {int(row['genre_country_combos'])}")

print("\n" + "=" * 70)


# 🌍 Шаг 5B. Страна-жанр матрица

print("\n🌍 Анализ связи страны и жанра")
print("=" * 70)

# === 1. Кросс-таблица: страна × жанр ===
print("\n📋 Кросс-таблица: количество произведений по странам и жанрам")
print("(показываем только ненулевые ячейки, топ-20 по частоте)")

# Создаём кросс-таблицу, удаляем пропуски
crosstab = pd.crosstab(
    df_music["country"].fillna("Unknown"),
    df_music["genre"].fillna("Unknown")
)

# Преобразуем в длинный формат для удобного вывода
crosstab_long = crosstab.stack().reset_index(name="count")
crosstab_long = crosstab_long[crosstab_long["count"] > 0].nlargest(20, "count")

print(f"\n{'Страна':<25} {'Жанр':<20} {'Кол-во':>8}")
print("-" * 55)
for _, row in crosstab_long.iterrows():
    print(f"{row['country']:<25} {row['genre']:<20} {row['count']:>8}")

# === 2. Доминирующий жанр по каждой стране ===
print("\n🎯 Доминирующий жанр в каждой стране (Топ-10 стран по числу записей):")

dominant_genres = df_music.groupby("country")["genre"].agg(
    lambda x: x.value_counts().idxmax() if x.value_counts().any() else "Unknown"
).reset_index(name="dominant_genre")

# Добавляем количество записей для контекста
country_counts = df_music["country"].value_counts().head(10).reset_index()
country_counts.columns = ["country", "total_records"]

# Объединяем и выводим
top_countries_dominant = country_counts.merge(dominant_genres, on="country", how="left")
for _, row in top_countries_dominant.iterrows():
    print(f"   • {row['country']:<25} → {row['dominant_genre']} ({row['total_records']} записей)")

# === 3. Глобальность жанров: в скольких странах представлен каждый жанр ===
print("\n🌐 Топ-10 самых 'глобальных' жанров (по числу стран):")

genre_countries = df_music.groupby("genre")["country"].nunique().sort_values(ascending=False).head(10)
for rank, (genre, n_countries) in enumerate(genre_countries.items(), 1):
    total_works = df_music[df_music["genre"] == genre]["work"].nunique()
    print(f"   {rank}. {genre:<20} → в {n_countries} странах ({total_works} произведений)")

# === 4. Разнообразие жанров по странам ===
print("\n🎨 Топ-10 стран по жанровому разнообразию (число уникальных жанров):")

country_genre_diversity = df_music.groupby("country")["genre"].nunique().sort_values(ascending=False).head(10)
for rank, (country, n_genres) in enumerate(country_genre_diversity.items(), 1):
    total_works = df_music[df_music["country"] == country]["work"].nunique()
    print(f"   {rank}. {country:<25} → {n_genres} жанров ({total_works} произведений)")

# === 5. Локальные жанры: встречаются только в одной стране ===
print("\n🔒 Топ-10 'локальных' жанров (встречаются только в 1 стране):")

local_genres = df_music.groupby("genre")["country"].nunique()
local_genres = local_genres[local_genres == 1].sort_values(ascending=False)

# Показываем топ-10 с указанием страны
if len(local_genres) > 0:
    for rank, genre in enumerate(local_genres.head(10).index, 1):
        country = df_music[df_music["genre"] == genre]["country"].mode().iloc[0]
        count = len(df_music[df_music["genre"] == genre])
        print(f"   {rank}. {genre:<25} → только в {country} ({count} произведений)")
else:
    print("   ⚠️ Нет жанров, встречающихся только в одной стране")

print("\n✅ Анализ страна-жанр матрицы завершён")

🎵 Анализ продуктивности авторов

🏆 Топ-10 самых продуктивных авторов (по числу произведений):
   1. Франтишек Ладислав Челаковский: 69 произведений
   2. Роберт Бёрнс: 51 произведений
   3. Леош Яначек: 42 произведений
   4. Людвик Куба: 33 произведений
   5. Франтишек Бартош: 21 произведений
   6. Werner Theunissen: 21 произведений
   7. Карел Яромир Эрбен: 16 произведений
   8. Адам Вацлав Михна: 16 произведений
   9. Фредди Куинн: 13 произведений
   10. Нил Янг: 12 произведений

🔍 Детальная статистика для топ-10 авторов:
----------------------------------------------------------------------

✍️  Франтишек Ладислав Челаковский
   📚 Произведений: 69
   🎭 Жанров: 0 | 🌍 Стран: 1
   ⏱️ Длительность: ср. 0.0, медиана 0.0
   🔗 Уникальных комбинаций жанр+страна: 0

✍️  Роберт Бёрнс
   📚 Произведений: 51
   🎭 Жанров: 0 | 🌍 Стран: 1
   ⏱️ Длительность: ср. 0.0, медиана 0.0
   🔗 Уникальных комбинаций жанр+страна: 0

✍️  Леош Яначек
   📚 Произведений: 42
   🎭 Жанров: 0 | 🌍 Стран: 2
   ⏱️ Длител

/tmp/ipykernel_410/1648981939.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  author_stats = df_music.groupby("author").apply(get_author_stats).reset_index()


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨